# Measuring Particle Diffusion with a JEPA world model

<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
<a href="https://colab.research.google.com/github/DeepTrackAI/DeepLearningCrashCourse/blob/main/Companion//main/pinn/pinn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<strong>If using Colab/Kaggle:</strong> You need to uncomment the code in the cell below this one.
</div>

In [10]:
# !pip install deeptrack deeplay torch torchvision matplotlib scikit-learn  # Uncomment if using Colab/Kaggle.

Joint Embedding Predictive Architectures (JEPAs) allow an AI system to learn how the world works purely by observing it, creating an internal "world model" without requiring human labels.

In this notebook, you will use a JEPA world model to analyze a stochastic physical system: the Brownian diffusion of a particle. You will see how predicting in an abstract latent space—rather than predicting raw pixels—lets a neural network represent the statistics of an inherently unpredictable process, instead of chasing an exact future it has no way of knowing. By the end, you will test whether the model's representation captures something physically meaningful, by training a small linear probe to extract the particle's diffusion coefficient ($D$) straight from its abstract representation—a quantity the network was never directly trained to predict.

<div style="background-color: #f0f8ff; border: 2px solid #4682b4; padding: 10px;">
<strong>Note:</strong> This companion example extends several concepts introduced throughout the book, specifically, encoder-decoder architectures (Chapter 4) and particle diffusion (Chapter 11). Unlike several examples in the book, the network is trained not to reproduce its input but to predict its own future latent representations, learning an implicit model of the system's dynamics directly from simulated video.

**Deep Learning Crash Course**  
Giovanni Volpe, Benjamin Midtvedt, Jesús Pineda, Henrik Klein Moberg, Harshith Bachimanchi, Joana B. Pereira, Carlo Manzo  
No Starch Press, San Francisco (CA), 2026  
ISBN-13: 9781718503922  

[https://nostarch.com/deep-learning-crash-course](https://nostarch.com/deep-learning-crash-course)

You can find the other notebooks on the [Deep Learning Crash Course GitHub page](https://github.com/DeepTrackAI/DeepLearningCrashCourse).
</div>

## Understanding JEPA World Models
Many artificial intelligence problems require an understanding of how a surrounding environment evolves. For instance, a robot navigating a room needs to anticipate collisions, a self-driving car needs to predict pedestrian movements, and an automated microscope needs to track how moving cells or particles spread over time.

To make sense of the world, humans and animals rely on an internal World Model—a mental simulator that uses past observations to predict future outcomes. In machine learning, building an effective world model generally involves balancing two competing realities:

1. Unpredictable Specifics: High-frequency, chaotic, or stochastic details where the exact future state is fundamentally uncertain (e.g., the exact, jittery trajectory of a single diffusing particle).

2. Predictable Statistics: Structural invariants or global properties that govern how that uncertainty behaves over time (e.g., the environmental diffusion coefficient, $D$).

Traditional predictive machine learning models typically try to predict the future down to the exact pixel. Given a video sequence, a generative network (like a standard Video Autoencoder or GAN) attempts to reconstruct subsequent frames pixel-by-pixel. However, in stochastic environments, predicting every single pixel is fundamentally a losing game. Because the exact path of a random particle cannot be known in advance, pixel-space models suffer from the "blurry image" problem—they mathematically average all possible futures, resulting in a faded, low-utility smudge.

Joint Embedding Predictive Architectures (JEPAs) solve this dilemma by changing where the prediction happens. Instead of training a network to generate future raw pixels, a JEPA passes the data through an encoder and performs its predictions entirely within an abstract representation space (latent space).

Crucially, this architecture does not magically look "through" the randomness to find a hidden, clean physical signal. In a system like diffusion, the randomness **is** the physical signal. Instead, the JEPA learns to represent the macroscopic statistics of the uncertainty.  By optimizing to predict future latent states without memorizing unpredictable pixel-level specifics, the model naturally captures how the system spreads globally, allowing us to accurately extract underlying properties like $D$—matching the realistic performance boundaries of a truly chaotic system.

### The Generative Dead End: Why Pixel Prediction Fails
Historically, the most intuitive way to build an AI world model from video data was to use generative modeling. Given a sequence of past video frames, a neural network is optimized to output the exact raw pixels of the subsequent frames.

While visually striking when successful, Yann LeCun argues that generative pixel-level prediction is a fundamental engineering bottleneck—and an unfeasible strategy for learning physics—for two major reasons:

- The Nuisance Variable Problem: A single pixel value can change drastically due to irrelevant factors like a shifting shadow, a camera sensor's grain, or background leaves rustling in the wind. Generative models waste immense computational capacity trying to reconstruct these high-frequency, non-essential "nuisance variables."

- The Multimodal Uncertainty Trap: In a stochastic or chaotic environment, the exact future cannot be perfectly known. If a particle is undergoing random thermal collisions, it has infinite possible paths. When a pixel-level model tries to handle multiple possible futures simultaneously, the mathematical average of those futures results in a blurred, low-utility average image—a faded smudge.

### The JEPA Paradigm: Moving to Representation Space
Joint Embedding Predictive Architectures (JEPAs) circumvent the flaws of generative modeling by changing where the prediction takes place. Instead of predicting the future in high-dimensional pixel space, a JEPA predicts the future in a lower-dimensional, abstract representation space (latent space).

The JEPA framework relies on a highly synchronized multi-network system:

- The Context Encoder: Takes a history of frames (e.g., a 10-frame window) and flattens them into an abstract latent state vector, $z_0$, summarizing everything important about the system's current state.

- The Target Encoder: Processes the actual future frames and extracts their true abstract latent representation, $z_1$. Crucially, this encoder does not backpropagate gradients directly; its weights are updated as a slow Exponential Moving Average (EMA) of the Context Encoder to serve as a stable, moving anchor.

- The Latent Predictor: Receives the present latent state $z_0$ and a time horizon conditioning parameter ($\delta$), and is tasked with guessing the future abstract state $\hat{z}_1$.

Because a JEPA is tasked with predicting abstract features rather than exact pixel locations, it naturally learns to discard individual pixel-level unpredictabilities. Instead, it captures the macroscopic statistics of the system's uncertainty. It learns to represent how space and uncertainty scale over time relative to environmental constants (like the diffusion coefficient, $D$).By bypassing the need to generate images, the network is free to focus entirely on learning the mathematical structure of the physical environment, creating an elegant, robust window into self-supervised physical common sense.

### The JEPA Optimization Objective: Loss and the Collapse Problem

In traditional generative world models, the loss function is simple: it is usually a Mean Squared Error (MSE) calculated between the predicted pixels and the true future pixels. The raw pixel grid acts as a natural anchor that prevents the model from doing anything lazy.

In a JEPA, however, there is **no pixel decoder**. The loss is calculated entirely within the abstract latent space:
$$
L_{\text{pred}} = \|\hat{z}_1 - z_1\|^2
$$

Where $\hat{z}_1$ is the predicted future representation and $z_1$ is the true future representation produced by the target encoder. This design poses a massive mathematical danger known as **Representation Collapse**.

### The Danger of Trivial Representations
Because both the context encoder and the target encoder are neural networks that we control, the model can discover a massive "shortcut" to drive the prediction error to zero: **it can learn to output a constant vector for every single frame.** If the encoders map every single image sequence—regardless of whether the particle is moving fast, slow, left, or right—to the exact same vector (e.g., $z = [0, 0, \dots, 0]$), then the predictor can simply output zero. The prediction error becomes exactly zero, but the representations are completely useless, carrying zero information about the physics of the environment.

To build a meaningful world model, we must force the latent representations to be highly informative while minimizing prediction error. JEPA achieves this by introducing explicit **anti-collapse regularization** adapted from self-supervised frameworks like **VICReg** (Variance-Invariance-Covariance Regularization).

### The Three Pillars of the JEPA Loss Function

To prevent collapse and force the model to capture the true underlying physics, the total loss function is broken down into three distinct mathematical objectives: **Invariance (Prediction)**, **Variance**, and **Covariance**.

$$\mathcal{L}_{\text{total}} = \alpha \mathcal{L}_{\text{pred}} + \beta \mathcal{L}_{\text{var}} + \gamma \mathcal{L}_{\text{cov}}$$

#### 1. The Invariance Loss ($\mathcal{L}_{\text{pred}}$)
This is the core predictive world-model objective. It minimizes the Mean Squared Error between the predicted future latent state $\hat{z}_1$ and the actual target latent state $z_1$:

$$\mathcal{L}_{\text{pred}} = \frac{1}{B}\sum_{i=1}^{B} \|\hat{z}_{1,i} - z_{1,i}\|^2$$

It forces the predictor to understand temporal dynamics. To minimize this, the model must calculate how a physical history transforms across an elapsed time horizon $\delta$.

#### 2. The Variance Regularizer ($\mathcal{L}_{\text{var}}$)
To prevent the encoders from collapsing into a single static point, the variance regularizer forces the latent vectors across a training batch to vary. It calculates the standard deviation $\sigma$ of each latent dimension across the batch and penalizes it if it drops below a target threshold (typically $1.0$):

$$\mathcal{L}_{\text{var}} = \frac{1}{d}\sum_{j=1}^{d} \max\left(0, 1 - \sigma(z_{\cdot, j})\right)$$

It acts as an expansive force. It explicitly forbids the encoders from squeezing all physical images into a single point, ensuring that different physical behaviors are mapped to distinct, unique locations in latent space.

#### 3. The Covariance Regularizer ($\mathcal{L}_{\text{cov}}$)
Even with high variance, a model can cheat by making all latent variables track the exact same signal (e.g., if dimension 1 tracks particle position, dimensions 2 through 16 might redundantly copy dimension 1). The covariance loss penalizes the off-diagonal elements of the latent covariance matrix:

$$\mathcal{L}_{\text{cov}} = \frac{1}{d}\sum_{j \neq k} \left(\text{Cov}(z)_{j,k}\right)^2$$

It acts as an information decoherer. It forces the different dimensions of your latent space to be linearly independent of one another. This maximizes the "capacity" of the embedding space, pushing the model to cleanly separate different orthogonal physical features (like tracking the particle's spatial $x,y$ position in some dimensions, and isolating the global diffusion rate $D$ in others).

## Simulating particle-diffusion videos

You'll simulate short video clips of a Brownian particle in a box, where the diffusion coefficient $D$ varies from
clip to clip. The world model will never be told $D$ directly — it has to infer it implicitly from how "jittery"
the particle's motion looks across frames. This is exactly the kind of latent physical parameter a JEPA-style model
should be able to recover from dynamics alone.

In [11]:
import numpy as np

IMAGE_SIZE = 64        # image size in pixels, corresponds to the box size 
N_FRAMES = 20          # frames per clip
DELTA_T = 1.0          # time between frames (arbitrary units)

def reflect(pos, lo, hi):
    """Reflect a scalar position back into [lo, hi] if it overshoots."""
    span = hi - lo
    pos = pos - lo
    pos = np.abs(pos)                                # reflect off lo
    pos = pos % (2 * span)
    pos = np.where(pos > span, 2 * span - pos, pos)  # reflect off hi
    return pos + lo

def simulate_trajectory(D, n_frames=N_FRAMES, image_size=IMAGE_SIZE,
                         delta_t=DELTA_T, margin=4):
    """Simulate a single Brownian trajectory with reflective boundaries.

    Returns:
        positions: (n_frames, 2) true (x, y) positions in pixel units
    """
    pos = np.array([image_size // 2, image_size // 2])  # start in the center
    # pos = np.random.uniform(margin, image_size - margin, size=2)
    positions = [pos.copy()]
    for _ in range(n_frames - 1):
        step = np.sqrt(2 * D * delta_t) * np.random.randn(2)
        pos = pos + step
        pos[0] = reflect(pos[0], margin, image_size - margin)
        pos[1] = reflect(pos[1], margin, image_size - margin)
        positions.append(pos.copy())
    return np.array(positions)

## Optical Rendering through a Fluorescence Microscope

In [12]:
import deeptrack as dt

def render_trajectory(positions, image_size=IMAGE_SIZE):
    """Render a sequence of (x, y) positions into a video of a fluorescent particle.

    Returns:
        frames: (n_frames, image_size, image_size) float32 array in [0, 1]
    """
    current_position = {"value": positions[0]}

    optics = dt.Fluorescence(
        NA=0.8,
        wavelength=560e-9,
        resolution=1e-7,
        magnification=1,
        output_region=(0, 0, image_size, image_size),
    )
    particle = dt.PointParticle(
        position=lambda: current_position["value"],
        position_unit="pixel",
        intensity=200,
    )
    pipeline = optics(particle)

    frames = []
    for p in positions:
        current_position["value"] = p
        frame = pipeline.update()()
        frames.append(np.asarray(frame).squeeze())
    frames = np.stack(frames).astype("float32")
    frames = frames / (frames.max() + 1e-8)
    return frames

In [13]:
def make_particle_clip(D, n_frames=N_FRAMES, image_size=IMAGE_SIZE, delta_t=DELTA_T):
    """Simulate and render one video clip of a single Brownian particle.

    Returns:
        frames: (n_frames, image_size, image_size) float32 array in [0, 1]
        positions: (n_frames, 2) true (x, y) positions in pixel units
    """
    positions = simulate_trajectory(D, n_frames, image_size, delta_t)
    frames = render_trajectory(positions, image_size)
    return frames, positions

### Visualize an Example Clip

In [14]:
from matplotlib import pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Quick sanity check
D_demo = 10.0
frames_demo, pos_demo = make_particle_clip(D_demo)

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(frames_demo[0], cmap="gray", vmin=frames_demo.min(), vmax=frames_demo.max())
title = ax.set_title(f"D={D_demo}, t=0")
ax.axis("off")

def update(i):
    im.set_data(frames_demo[i])
    title.set_text(f"D={D_demo}, t={i}")
    return [im, title]

anim = animation.FuncAnimation(
    fig, update, frames=len(frames_demo), interval=150, blit=False
)
plt.close(fig)  # prevent a duplicate static plot from also rendering

HTML(anim.to_jshtml())

## Data Pipeline: Constructing the Context and Target Windows

To train the JEPA world model, you'll need a dataset that streams video clips of diffusing particles.  The `DiffusionClipDataset()` handles this by dynamically generating video clips on the fly and slicing them into specific temporal windows. For every index sampled, the dataset performs the following operations:

- Simulates a Full Video: It draws a random environmental diffusion coefficient ($D$) from our specified range and generates a continuous trajectory of N_FRAMES.

- Establishes the Present Anchor: It randomly selects a frame index $t_0$ to represent the "present moment." To ensure there are enough past frames to fill our context window, $t_0$ is constrained to look back at least `window` frames.

- Samples a Random Future Horizon: It randomly samples a time gap ($\delta$) ranging from $1$ to the maximum remaining frames in the clip. This sets our future target frame at $t_1 = t_0 + \delta$.

- Assembles the Tensors:

    - x0 (Context Window): A sequence of 10 consecutive frames leading up to and including the present moment ($[t_0 - 9, \dots, t_0]$).
    - x1 (Target Window): A sequence of 10 consecutive frames leading up to and including the future moment ($[t_1 - 9, \dots, t_1]$).
    - Conditioning & Ground Truths: It extracts the elapsed time gap delta_t ($\Delta t$), the true environmental rate D_true ($D$), and the precise 2D spatial positions (pos0, pos1) of the particle at both timestamps for downstream verification.

In [15]:
import torch

WINDOW = 10 # window size for sampling clips
D_RANGE = (0.1, 10.0)  # range of diffusion coefficients to sample from

class DiffusionClipDataset(torch.utils.data.Dataset):
    def __init__(self, n_clips=2000, n_frames=N_FRAMES, image_size=IMAGE_SIZE,
                 d_range=D_RANGE, window=WINDOW):
        self.n_clips = n_clips
        self.n_frames = n_frames
        self.image_size = image_size
        self.d_range = d_range
        self.window = window

    def __len__(self):
        return self.n_clips

    def __getitem__(self, idx):
        D = np.random.uniform(*self.d_range)
        frames, positions = make_particle_clip(D, self.n_frames, self.image_size)

        w = self.window
        t0 = np.random.randint(w - 1, self.n_frames - w )
        delta = np.random.randint(w, self.n_frames - t0)
        t1 = t0 + delta

        x0 = torch.from_numpy(frames[t0 - w + 1 : t0 + 1]).float()  # (w, H, W)
        x1 = torch.from_numpy(frames[t1 - w + 1 : t1 + 1]).float()  # (w, H, W)

        delta_t = torch.tensor([delta], dtype=torch.float32)
        D_true = torch.tensor([D], dtype=torch.float32)
        pos0 = torch.from_numpy(positions[t0]).float()
        pos1 = torch.from_numpy(positions[t1]).float()

        return x0, x1, delta_t, D_true, pos0, pos1

train_ds = DiffusionClipDataset(n_clips=2000)
val_ds = DiffusionClipDataset(n_clips=400)

train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=32, shuffle=False)


## Understanding World Models

We will build a **Joint-Embedding Predictive Architecture (JEPA)**:
- an **encoder** $E_\theta$ that maps a video frame to a latent state $z_t = E_\theta(x_t)$
- a **target encoder** $E_{\bar\theta}$ (an EMA copy of $E_\theta$, no gradient) that produces the *prediction target*
- a **predictor** $P_\phi$ that predicts $\hat z_{t+\Delta t} = P_\phi(z_t, \Delta t)$
- training signal: $\hat z_{t+\Delta t} \approx E_{\bar\theta}(x_{t+\Delta t})$, **not** pixel reconstruction

The key pedagogical point: we never ask the model to reconstruct pixels. Diffusion videos are mostly noise/texture —
not worth modeling. We ask the model to predict *its own representation* of the future. We will see this is harder to
get right (it can collapse to a trivial constant) and we'll fix that with an EMA target + a variance regularizer
(VICReg-style), in the spirit of I-JEPA / V-JEPA.

### Roadmap
1. Simulate particle-diffusion videos with **DeepTrack2** (vary the diffusion coefficient $D$)
2. Build encoder / target-encoder / predictor
3. Train self-supervised with a latent-prediction loss + anti-collapse regularization
4. **Probe**: train a tiny linear head latent → true $D$, true position — this is the moment we check whether the
   world model "discovered" physics
5. Visualize latent trajectories vs. true trajectories
6. **Ablation**: remove the EMA target / regularizer and watch the representation collapse
7. (Optional, advanced) Differentiable-simulator comparison: optimize $D$ directly through DeepTrack2's
   gradient-preserving pipeline and compare to what the learned world model infers

## 3. The world model: encoder, EMA target encoder, predictor

We build everything with **Deeplay**, so each block is a swappable, composable module. The encoder is a small CNN;
the predictor is an MLP that takes $(z_t, \Delta t)$ and outputs $\hat z_{t+\Delta t}$.

Two design choices worth flagging:
- **EMA target encoder**: the target representation $E_{\bar\theta}(x_{t+\Delta t})$ is produced by a *momentum copy*
  of the encoder, updated as $\bar\theta \leftarrow \tau \bar\theta + (1-\tau)\theta$, with **no gradient** flowing
  through it. This is the standard trick (BYOL/I-JEPA/V-JEPA) to prevent the trivial collapse "encoder outputs a
  constant, predictor learns the constant, loss = 0".
- **Variance regularization** (VICReg-style): we additionally penalize the embeddings if their per-dimension
  variance across the batch collapses toward 0. This is a second, complementary defense against collapse, and lets
  us demonstrate what happens when we strip each one out (Section 6).


## Framing the Architecture: Why We Condition on Time ($\Delta t$)

When applying Joint Embedding Predictive Architectures to video data (such as Meta's V-JEPA), it is common practice to use fixed-size context and target windows. For example, a model might take a fixed block of frames and learn to predict a subsequent, fixed block of frames. Because the time gap between the context and the target is always identical, the predictor network does not need to know when it is predicting; it only needs to learn a static temporal mapping.

However, because our goal is to recover the underlying physics of diffusion, we introduce a deliberate modification to the standard literature setup: **we explicitly condition our Latent Predictor on a variable time horizon ($\Delta t$).**

Instead of predicting a single fixed future block, our model is given a 10-frame context window and asked to predict anywhere from 1 to 14 frames into the future, with the exact horizon sampled randomly for every training example.

### The Pedagogical Value of Variable Horizons

We make this architectural departure for two reasons:

1. **Discouraging a "Memorized Displacement" Shortcut:** In pure Brownian motion, the environmental diffusion coefficient ($D$) is not a static displacement; it is the proportionality constant that dictates how the uncertainty scales over time ($\sigma^2 \sim 2D\Delta t$). If the time gap were always fixed, the model could satisfy the prediction objective by memorizing a one-off displacement scale for that specific horizon, without ever needing a notion of rate. By varying $\Delta t$, the model is instead asked to be consistent across many different horizons simultaneously—a design intended to encourage it to internalize a generalizable rate, rather than a single fixed-horizon shortcut.
2. **Visualizing the Arrow of Diffusion:** Conditioning on $\Delta t$ lets us evaluate the model's performance as a function of elapsed time. This makes it possible to directly plot how latent prediction error grows as the horizon expands—a quantifiable window into how a world model handles accumulating stochastic uncertainty.

In [ ]:
import deeplay as dl
from typing import Optional
from torch import nn
import torch.nn.functional as F


LATENT_DIM = 64

class WorldModel(dl.Application):
    def __init__(self, latent_dim=LATENT_DIM, ema_tau=0.99, lam=1.0, mu=1.0, nu=0.01, use_ema=True,
                 optimizer=None, **kwargs):

        self.encoder = dl.ConvolutionalEncoder2d(
            in_channels=WINDOW,
            hidden_channels=[32, 64],
            out_channels=128,
            postprocess=dl.Layer(nn.AdaptiveAvgPool2d, 1),
        )
        self.encoder.strided(stride=2, apply_to_first_layer=True, apply_to_last_layer=True)
        self.encoder_proj = nn.Sequential(
            nn.Linear(128, latent_dim),
            nn.BatchNorm1d(latent_dim),
        )

        self.predictor = dl.MultiLayerPerceptron(
            in_features=latent_dim + 1,
            hidden_features=[128, 128],
            out_features=latent_dim,
        )
        self.predictor["blocks", :-1].all.normalized(nn.BatchNorm1d)
        self.predictor["blocks", :-1].configure(order=["layer", "normalization", "activation"])


        self.use_ema = use_ema
        self.ema_tau = ema_tau
        self.lam = lam
        self.mu = mu
        self.nu = nu

        if self.use_ema:
            self.target_encoder = dl.ConvolutionalEncoder2d(
                in_channels=WINDOW,
                hidden_channels=[32, 64],
                out_channels=128,
                postprocess=dl.Layer(nn.AdaptiveAvgPool2d, 1),
            )
            self.target_encoder.strided(stride=2, apply_to_first_layer=True, apply_to_last_layer=True)
            self.target_proj = nn.Sequential(
                nn.Linear(128, latent_dim),
                nn.BatchNorm1d(latent_dim),
            )
        else:
            self.target_encoder = self.encoder
            self.target_proj = self.encoder_proj

        super().__init__(**kwargs)

        self.optimizer = optimizer or dl.Adam(lr=1e-4)

        @self.optimizer.params
        def params(self):
            return self.parameters()

        self._target_synced = False

    def encode(self, x):
        z = self.encoder_proj(self.encoder(x).flatten(1))
        return F.normalize(z, dim=-1)

    def encode_target(self, x):
        z = self.target_proj(self.target_encoder(x).flatten(1))
        return F.normalize(z, dim=-1)

    def forward(self, x0, x1, delta_t):
        z0 = self.encode(x0)
        z1_pred = self.predictor(torch.cat([z0, delta_t / N_FRAMES], dim=-1))
        z1_pred = F.normalize(z1_pred, dim=-1)   # predictor output must match target's scale
        if self.use_ema:
            with torch.no_grad():
                z1_target = self.encode_target(x1)
        else:
            z1_target = self.encode_target(x1)
        return z0, z1_pred, z1_target
    
    def variance_loss(self, z, gamma=None):
        if gamma is None:
            gamma = 1.0 / (z.shape[-1] ** 0.5)   # ~0.125 for latent_dim=64
        std = z.std(dim=0) + 1e-4
        return F.relu(gamma - std).mean()

    # def covariance_loss(self, z):
    #     z = z - z.mean(dim=0)
    #     n, d = z.shape
    #     cov = (z.T @ z) / (n - 1)
    #     off_diag = cov.flatten()[:-1].view(d - 1, d + 1)[:, 1:].flatten()
    #     return (off_diag ** 2).sum() / d
    
    def covariance_loss(self, z):
        z = z - z.mean(dim=0)
        cov = (z.T @ z) / (z.shape[0] - 1)
        off_diag = cov - torch.diag(torch.diagonal(cov))
        return (off_diag ** 2).sum() / z.shape[1]

    def compute_loss(self, z0, z1_pred, z1_target):
        pred_loss = F.mse_loss(z1_pred, z1_target)
        loss = {"pred": self.lam * pred_loss}
        if self.mu > 0:
            reg_loss = self.variance_loss(z0) + self.variance_loss(z1_pred)
            loss["reg"] = self.mu * reg_loss
        if self.nu > 0:
            cov_loss = self.covariance_loss(z0) + self.covariance_loss(z1_pred)
            loss["cov"] = self.nu * cov_loss
        return loss

    def _shared_step(self, batch, stage):
        x0, x1, delta_t, D_true, pos0, pos1 = batch
        z0, z1_pred, z1_target = self(x0, x1, delta_t)
        loss = self.compute_loss(z0, z1_pred, z1_target)
        for name, v in loss.items():
            self.log(f"{stage}_{name}", v, on_step=True, on_epoch=True,
                      prog_bar=True, logger=True)
        return sum(loss.values())

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "val")

    def on_train_batch_end(self, outputs, batch, batch_idx):
        if self.use_ema:
            self.update_target()

    # @torch.no_grad()
    # def update_target(self):
    #     if not self._target_synced:
    #         self.target_encoder.load_state_dict(self.encoder.state_dict())

    @torch.no_grad()
    def update_target(self):
        if not self._target_synced:
            self.target_encoder.load_state_dict(self.encoder.state_dict())
            self.target_proj.load_state_dict(self.encoder_proj.state_dict())
            for p in list(self.target_encoder.parameters()) + list(self.target_proj.parameters()):
                p.requires_grad_(False)
            self._target_synced = True
            return
        for p, p_t in zip(self.encoder.parameters(), self.target_encoder.parameters()):
            p_t.data.mul_(self.ema_tau).add_(p.data, alpha=1 - self.ema_tau)
        for p, p_t in zip(self.encoder_proj.parameters(), self.target_proj.parameters()):
            p_t.data.mul_(self.ema_tau).add_(p.data, alpha=1 - self.ema_tau)

In [ ]:
from lightning.pytorch.callbacks import Callback
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

class ProbeMonitorCallback(Callback):
    def __init__(self, check_every_n_epochs=1, n_clips=300):
        self.check_every_n_epochs = check_every_n_epochs
        self.n_clips = n_clips
        self.history = []

    @torch.no_grad()
    def _collect_probe_data(self, model):
        model.eval()
        Z, Z2, DT, DS, POS = [], [], [], [], []
        ds = DiffusionClipDataset(n_clips=self.n_clips)
        loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)
        for x0, x1, delta_t, D_true, pos0, pos1 in loader:
            x0, x1 = x0.to(model.device), x1.to(model.device)
            z0 = model.encode(x0)
            z1 = model.encode(x1)
            Z.append(z0.cpu()); Z2.append(z1.cpu())
            DT.append(delta_t); DS.append(D_true); POS.append(pos0)
        return torch.cat(Z), torch.cat(Z2), torch.cat(DT), torch.cat(DS), torch.cat(POS)

    def on_train_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch
        if (epoch + 1) % self.check_every_n_epochs != 0:
            return

        Z0, Z1, DT_, D_, POS0 = self._collect_probe_data(pl_module)
        feat_D = torch.cat([Z0, Z1, DT_], dim=1).numpy()
        target_D = D_.numpy().ravel()

        Xtr, Xte, ytr, yte = train_test_split(feat_D, target_D, test_size=0.25, random_state=0)
        probe = Ridge(alpha=1.0).fit(Xtr, ytr)
        r2 = r2_score(yte, probe.predict(Xte))

        self.history.append((epoch, r2))
        pl_module.log("probe_D_r2", r2, prog_bar=True, on_step=False, on_epoch=True)
        print(f"  [epoch {epoch}] D probe R^2 = {r2:.3f}")

        pl_module.train()

In [ ]:
probe_monitor = ProbeMonitorCallback(check_every_n_epochs=1, n_clips=300)

model = WorldModel(optimizer=dl.Adam(lr=1e-4)).create()
history = model.fit(train_ds, val_data=val_ds, max_epochs=15, batch_size=32, accelerator = "auto", callbacks=[probe_monitor])

┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder        │ ConvolutionalEncoder2d │ 95.3 K │ train │     0 │
│ 1 │ encoder_proj   │ Sequential             │  8.4 K │ train │     0 │
│ 2 │ predictor      │ MultiLayerPerceptron   │ 33.7 K │ train │     0 │
│ 3 │ target_encoder │ ConvolutionalEncoder2d │ 95.3 K │ train │     0 │
│ 4 │ target_proj    │ Sequential             │  8.4 K │ train │     0 │
│ 5 │ train_metrics  │ MetricCollection       │      0 │ train │     0 │
│ 6 │ val_metrics    │ MetricCollection       │      0 │ train │     0 │
│ 7 │ test_metrics   │ MetricCollection       │      0 │ train │     0 │
│ 8 │ optimizer      │ Adam                   │      0 │ train │     0 │
└───┴────────────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 241 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 241 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 47                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/cmanzo/Documents/GitHub/Environments/deeptrack_dev/lib/python3.12/site-packages/lightning/pytorch/utilities/
_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

/Users/cmanzo/Documents/GitHub/Environments/deeptrack_dev/lib/python3.12/site-packages/lightning/pytorch/trainer/co
nnectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider
increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.

...

[epoch 1] D probe R^2 = 0.668

[epoch 2] D probe R^2 = 0.445

[epoch 3] D probe R^2 = 0.536

[epoch 4] D probe R^2 = 0.566

[epoch 5] D probe R^2 = 0.656

In [ ]:
epochs, r2s = zip(*probe_monitor.history)
plt.plot(epochs, r2s, marker="o")
plt.xlabel("epoch"); plt.ylabel("D probe R²")
plt.title("Probe R² over training")
plt.show()

## 4. Loss: latent prediction + anti-collapse regularizer

$$ \mathcal{L} = \underbrace{\|\hat z_{t+\Delta t} - \text{sg}(z_{t+\Delta t})\|^2}_{\text{prediction loss}} \;+\; \lambda \underbrace{\sum_d \max(0,\, \gamma - \text{std}(z_{\cdot, d}))}_{\text{variance regularizer (VICReg-style)}} $$

`sg` = stop-gradient (handled here by the target encoder having `requires_grad=False` and being updated only via EMA).
The variance term pushes each latent dimension to keep some spread *within the batch*, so it can't collapse to a
single point for every input.


## 6. Probing: did the world model discover physics?

We freeze the encoder and fit a small linear/MLP probe from $z_t$ alone to:
- the true diffusion coefficient $D$ (a *global* property of the clip)
- the true particle position (a property of the single frame)

If the encoder's latent space is rich enough to predict $D$ well, the self-supervised prediction objective has
forced the model to represent something about the *dynamics regime* of the clip — not just the current pixel
pattern. This is the chapter's main "aha" moment.

Note: $D$ is a property of the *whole clip*, not a single frame, so to probe it fairly we feed the probe a short
window of frames (or, more simply here, two latents $z_t, z_{t+\Delta t}$ and let it use their difference).


# diagnosstic 1

In [ ]:
## Diagnostic: is D actually recoverable from raw pixel displacements?
#
# This bypasses the model entirely. We simulate many clips at different D,
# compute the raw pixel displacement statistics directly from the rendered
# frames (via simple centroid tracking, NOT from the ground-truth `positions`
# array -- we want to know what's visible in the IMAGES, since that's all the
# encoder ever sees), and check whether D is recoverable from that signal.
#
# If this comes back with a weak/no relationship, the bottleneck is the
# rendering/resolution regime, not the world-model architecture, and no
# amount of context_k or loss tuning will fix it -- you'd need to change the
# image_size, the D range, or the optics (PSF size, magnification, etc.).

import numpy as np
import matplotlib.pyplot as plt


def centroid_from_frame(frame, threshold_rel=0.3):
    """Cheap centroid estimate directly from pixel intensities (mimics what
    any reasonable encoder could in principle extract from a single frame)."""
    thresh = frame.max() * threshold_rel
    mask = frame > thresh
    if mask.sum() == 0:
        mask = frame > 0
    ys, xs = np.nonzero(mask)
    weights = frame[ys, xs]
    if weights.sum() == 0:
        return np.array([np.nan, np.nan])
    cy = np.average(ys, weights=weights)
    cx = np.average(xs, weights=weights)
    return np.array([cx, cy])


def measure_pixel_displacement_vs_D(n_clips=200, k=14, n_frames=N_FRAMES):
    Ds, true_disps, pix_disps = [], [], []

    for _ in range(n_clips):
        D = np.random.uniform(*D_RANGE)
        frames, positions = make_particle_clip(D, n_frames=n_frames)

        t0s = np.random.randint(0, n_frames - k, size=3)
        for t0 in t0s:
            true_disp = np.linalg.norm(positions[t0 + k] - positions[t0])

            c0 = centroid_from_frame(frames[t0])
            c1 = centroid_from_frame(frames[t0 + k])
            if np.any(np.isnan(c0)) or np.any(np.isnan(c1)):
                continue
            pix_disp = np.linalg.norm(c1 - c0)

            Ds.append(D)
            true_disps.append(true_disp)
            pix_disps.append(pix_disp)

    return np.array(Ds), np.array(true_disps), np.array(pix_disps)


Ds, true_disps, pix_disps = measure_pixel_displacement_vs_D(n_clips=200, k=14)

print(f"n samples: {len(Ds)}")
print(f"correlation(D, true_disp)  = {np.corrcoef(Ds, true_disps)[0, 1]:.3f}  "
      f"(sanity check -- should be strongly positive by construction)")
print(f"correlation(D, pix_disp)   = {np.corrcoef(Ds, pix_disps)[0, 1]:.3f}  "
      f"(THIS is what matters -- can pixels alone reveal D?)")
print(f"correlation(true_disp, pix_disp) = {np.corrcoef(true_disps, pix_disps)[0, 1]:.3f}  "
      f"(how faithfully does rendering preserve the true displacement?)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(Ds, true_disps, s=8, alpha=0.4)
axes[0].set_xlabel("D"); axes[0].set_ylabel("true displacement (k frames)")
axes[0].set_title("Ground truth: D vs true displacement")

axes[1].scatter(Ds, pix_disps, s=8, alpha=0.4, color="orange")
axes[1].set_xlabel("D"); axes[1].set_ylabel("pixel-centroid displacement (k frames)")
axes[1].set_title("From RENDERED PIXELS: D vs measured displacement")

axes[2].scatter(true_disps, pix_disps, s=8, alpha=0.4, color="green")
lims = [0, max(true_disps.max(), pix_disps.max())]
axes[2].plot(lims, lims, "r--", alpha=0.5)
axes[2].set_xlabel("true displacement"); axes[2].set_ylabel("pixel-centroid displacement")
axes[2].set_title("Rendering fidelity check")

plt.tight_layout()
plt.show()

# diagnostic

In [ ]:
import matplotlib.pyplot as plt

train_pred = history.history["train_pred_epoch"]["value"]
train_reg = history.history.get("train_reg_loss_val")

plt.figure(figsize=(6, 4))
plt.plot(train_pred, label="pred_loss")
if train_reg is not None:
    plt.plot(train_reg, label="reg_loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("Training loss breakdown")
plt.show()

print(f"pred_loss: first={train_pred[0]:.4f}  last={train_pred[-1]:.4f}")
print(f"  -> dropped to {100 * train_pred[-1] / train_pred[0]:.1f}% of initial value")

In [ ]:
import torch

@torch.no_grad()
def check_latent_variance(model, n_clips=300):
    model.eval()
    ds = DiffusionClipDataset(n_clips=n_clips)
    loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)
    Zs = []
    for x0, x1, delta_t, D_true, pos0, pos1 in loader:
        x0 = x0.to(model.device)
        Zs.append(model.encode(x0).cpu())
    Z = torch.cat(Zs)
    return Z, Z.std(dim=0)

Z0_check, per_dim_std = check_latent_variance(model)
print(f"Per-dim std -- mean: {per_dim_std.mean():.4f}, min: {per_dim_std.min():.4f}, max: {per_dim_std.max():.4f}")

gamma = 1.0  # whatever you set in variance_loss
near_floor = (per_dim_std - gamma).abs() < 0.1
print(f"Dims near the regularizer floor: {near_floor.sum().item()} / {len(per_dim_std)}")

# Confirm it's not just stochastic noise in eval mode
x0_sample, *_ = next(iter(torch.utils.data.DataLoader(DiffusionClipDataset(n_clips=4), batch_size=4)))
x0_sample = x0_sample.to(model.device)
z_a, z_b = model.encode(x0_sample), model.encode(x0_sample)
print(f"Repeat-forward diff (should be ~0): {(z_a - z_b).abs().max().item():.6f}")

In [ ]:
from sklearn.decomposition import PCA
pca = PCA().fit(Z0_check.numpy())
print(np.cumsum(pca.explained_variance_ratio_)[:10])

# test

In [ ]:
@torch.no_grad()
def collect_probe_data(model, n_clips=600):
    model.eval()
    Z, Z2, DT, DS, POS = [], [], [], [], []
    ds = DiffusionClipDataset(n_clips=n_clips)
    loader = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=False)
    for x0, x1, delta_t, D_true, pos0, pos1 in loader:
        x0, x1 = x0.to(model.device), x1.to(model.device)
        z0 = model.encode(x0)
        z1 = model.encode(x1)
        Z.append(z0.cpu()); Z2.append(z1.cpu())
        DT.append(delta_t); DS.append(D_true); POS.append(pos0)
    return (torch.cat(Z), torch.cat(Z2), torch.cat(DT), torch.cat(DS), torch.cat(POS))


Z0, Z1, DT_, D_, POS0 = collect_probe_data(model)

# Feature for the D-probe: concatenate z0, z1, and delta_t (so the probe can use
# "how much did the latent move, given how much time passed" -- exactly the
# quantity that defines a diffusion coefficient).
feat_D = torch.cat([Z0, Z1, DT_], dim=1).numpy()
target_D = D_.numpy().ravel()

feat_pos = Z0.numpy()
target_pos = POS0.numpy()

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

Xtr, Xte, ytr, yte = train_test_split(feat_D, target_D, test_size=0.25, random_state=0)
probe_D = Ridge(alpha=1.0).fit(Xtr, ytr)
pred_D = probe_D.predict(Xte)
print(f"D probe  R^2 = {r2_score(yte, pred_D):.3f}")

Xtr2, Xte2, ytr2, yte2 = train_test_split(feat_pos, target_pos, test_size=0.25, random_state=0)
probe_pos = Ridge(alpha=1.0).fit(Xtr2, ytr2)
pred_pos = probe_pos.predict(Xte2)
print(f"position probe  R^2 = {r2_score(yte2, pred_pos):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(yte, pred_D, s=10, alpha=0.5)
axes[0].plot([yte.min(), yte.max()], [yte.min(), yte.max()], "r--")
axes[0].set_xlabel("true D"); axes[0].set_ylabel("predicted D"); axes[0].set_title("D probe")

axes[1].scatter(yte2[:, 0], pred_pos[:, 0], s=10, alpha=0.5, label="x")
axes[1].scatter(yte2[:, 1], pred_pos[:, 1], s=10, alpha=0.5, label="y")
axes[1].plot([0, IMAGE_SIZE], [0, IMAGE_SIZE], "r--")
axes[1].set_xlabel("true position"); axes[1].set_ylabel("predicted position")
axes[1].legend(); axes[1].set_title("position probe")
plt.tight_layout(); plt.show()


## 7. Visualizing latent trajectories

A qualitative check: encode every frame of a single clip and look at the latent trajectory (via PCA), next to the
true (x, y) trajectory. A good world model's latent trajectory should "look like" a (possibly distorted/rotated)
version of the true motion — smooth, continuous, and varying systematically with $D$.


In [ ]:
from sklearn.decomposition import PCA

@torch.no_grad()
# def encode_full_clip(model, D):
#     frames, positions = make_particle_clip(D)
#     x = torch.from_numpy(frames).unsqueeze(1).to(model.device)  # (T, 1, H, W)
#     z = model.encode(x).cpu().numpy()   # <-- was model.encoder(x)
#     return z, positions
def encode_full_clip(model, D, w=WINDOW):
    frames, positions = make_particle_clip(D)
    windows = np.stack([frames[i - w + 1 : i + 1] for i in range(w - 1, len(frames))])
    x = torch.from_numpy(windows).float().to(model.device)
    z = model.encode(x).cpu().numpy()
    return z, positions[w - 1:]


model.eval()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, D_val in zip(axes, [0.1, 0.8, 1.8]):
    z, positions = encode_full_clip(model, D_val)
    z_pca = PCA(n_components=2).fit_transform(z)
    ax.plot(positions[:, 0], positions[:, 1], "o-", label="true (x, y)", alpha=0.6)
    ax2 = ax.twinx().twiny()
    ax2.plot(z_pca[:, 0], z_pca[:, 1], "x--", color="orange", label="latent (PCA)", alpha=0.8)
    ax.set_title(f"D = {D_val}")
plt.tight_layout(); plt.show()


## 8. Ablation: what happens without the anti-collapse defenses?

We retrain two broken variants:
- **No EMA target** (predictor and target encoder share weights and gradients — a classic recipe for collapse)
- **No variance regularizer** ($\lambda = 0$)

Watch the prediction loss: it can go to (near) zero *for the wrong reason* — the encoder learns to output a
near-constant vector, which is trivially easy to "predict". The probe R² is the tell: collapse gives low prediction
loss but a useless representation (probe R² near zero).


In [ ]:
class WorldModelNoEMA(WorldModel):
    """Target encoder IS the online encoder (no separate weights, no stop-gradient)."""
    def forward(self, x0, x1, delta_t):
        z0 = self.encoder(x0)
        z1_pred = self.predictor(z0, delta_t)
        z1_target = self.encoder(x1)  # gradient flows here too -- no stop-gradient!
        return z0, z1_pred, z1_target

    @torch.no_grad()
    def update_target(self):
        pass  # nothing to update; there is no separate target encoder


def quick_eval_probe(model, n_clips=400):
    Z0, Z1, DT_, D_, POS0 = collect_probe_data(model, n_clips=n_clips)
    feat_D = torch.cat([Z0, Z1, DT_], dim=1).numpy()
    Xtr, Xte, ytr, yte = train_test_split(feat_D, D_.numpy().ravel(), test_size=0.25, random_state=0)
    probe = Ridge(alpha=1.0).fit(Xtr, ytr)
    return r2_score(yte, probe.predict(Xte))


# Variant A: no EMA target (collapse-prone)
model_no_ema = WorldModelNoEMA().to(device)
hist_no_ema = train_world_model(model_no_ema, train_loader, val_loader, n_epochs=10, lam=1.0)
r2_no_ema = quick_eval_probe(model_no_ema)
print(f"No-EMA variant: final pred_loss={hist_no_ema['pred_loss'][-1]:.4f}, probe R^2={r2_no_ema:.3f}")

# Variant B: no variance regularizer (lam=0), EMA kept
model_no_reg = WorldModel().to(device)
hist_no_reg = train_world_model(model_no_reg, train_loader, val_loader, n_epochs=10, lam=0.0)
r2_no_reg = quick_eval_probe(model_no_reg)
print(f"No-regularizer variant: final pred_loss={hist_no_reg['pred_loss'][-1]:.4f}, probe R^2={r2_no_reg:.3f}")

print(f"Full model probe R^2 was: {r2_score(yte, pred_D):.3f}")
print("Compare: low pred_loss + low probe R^2 == collapse, not a good world model.")


## 9. (Optional, advanced) Analysis-by-synthesis through the differentiable simulator

DeepTrack2's pipeline is gradient-preserving. As a contrast to the *learned* world model, we can directly optimize a
diffusion coefficient by backpropagating an image-reconstruction loss through the simulator itself ("analysis by
synthesis" / differentiable rendering), and compare the result to what our learned model's probe infers from the
same clip. This connects the chapter back to your earlier "classical differentiable simulation" material and shows
two different routes to the same physical insight: a model that *learned* to infer $D$ implicitly, vs. optimization
that infers $D$ explicitly via a differentiable forward model.

We leave this as an extension for the reader / next revision of the notebook — it requires exposing $D$ as a
`torch.nn.Parameter` inside the simulation step rather than a plain numpy float, which depends on how your local
DeepTrack2/Deeplay version exposes differentiable parameters in `pipeline.update()`.


## 10. Summary & what to try next

- We trained a JEPA-style world model that predicts **latent** futures, not pixels, on simulated diffusion videos.
- The EMA target + variance regularizer were both necessary to avoid collapse — the ablation made this concrete.
- A simple linear probe shows the latent space encodes the diffusion coefficient $D$ and the particle's position,
  even though neither was ever a training target.

**Extensions worth trying:**
- Multiple particles per clip → forces a decision between a single global latent vs. per-particle "slots" (a natural
  segue into object-centric / slot-based world models)
- Add DeepTrack2's optical aberrations/noise to make the rendering more realistic, and see whether the probe R² for
  $D$ degrades — a nice lesson on how much "physics signal" survives realistic imaging noise
- Replace the CNN encoder with a small ViT (swap-in, thanks to Deeplay's modularity) and compare probe quality
- Try predicting multiple $\Delta t$ steps ahead recurrently, and see how prediction error grows with horizon —
  this is the classic compounding-error problem in world models
